# MTG Inventory Tool Walkthrough

This notebook is an onboarding tour for people who are new to the repo.

It walks through the core workflow end to end:

1. creating a clean SQLite database
2. importing small sample catalog and pricing files
3. creating a personal inventory and searching the local catalog
4. adding and editing owned cards
5. reviewing inventory health, exports, and reports
6. restoring the database from a safety snapshot after a destructive change

The notebook uses tiny local sample payloads so it is fast to rerun and safe to experiment with.


## Setup

Run the notebook from top to bottom so each section starts from a clean scratch database.

The setup cell below:

- finds the repo root
- imports the local package directly from `src/`
- creates a scratch workspace under `var/walkthrough/`

That keeps the walkthrough self-contained and avoids depending on any previously installed copy of the package.


In [ ]:
from __future__ import annotations

import importlib
import json
import os
import shutil
import sqlite3
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "mtg_source_stack").exists():
            return candidate
    raise RuntimeError("Could not find the repo root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / "src"
VAR_DIR = REPO_ROOT / "var"

SRC_DIR_STR = str(SRC_DIR)
sys.path = [path for path in sys.path if path != SRC_DIR_STR]
sys.path.insert(0, SRC_DIR_STR)

for module_name in list(sys.modules):
    if module_name == "mtg_source_stack" or module_name.startswith("mtg_source_stack."):
        del sys.modules[module_name]

CLI_ENV = os.environ.copy()
existing_pythonpath = CLI_ENV.get("PYTHONPATH")
CLI_ENV["PYTHONPATH"] = (
    f"{SRC_DIR}{os.pathsep}{existing_pythonpath}" if existing_pythonpath else str(SRC_DIR)
)

mvp_importer = importlib.import_module("mtg_source_stack.mvp_importer")
personal_inventory_cli = importlib.import_module("mtg_source_stack.personal_inventory_cli")

create_database_snapshot = mvp_importer.create_database_snapshot
initialize_database = mvp_importer.initialize_database
import_mtgjson_identifiers = mvp_importer.import_mtgjson_identifiers
import_mtgjson_prices = mvp_importer.import_mtgjson_prices
import_scryfall_cards = mvp_importer.import_scryfall_cards
list_database_snapshots = mvp_importer.list_database_snapshots
restore_database_snapshot = mvp_importer.restore_database_snapshot

add_card = personal_inventory_cli.add_card
create_inventory = personal_inventory_cli.create_inventory
export_inventory_csv = personal_inventory_cli.export_inventory_csv
format_add_card_result = personal_inventory_cli.format_add_card_result
format_export_csv_result = personal_inventory_cli.format_export_csv_result
format_inventory_health_result = personal_inventory_cli.format_inventory_health_result
format_inventory_report_result = personal_inventory_cli.format_inventory_report_result
format_merge_rows_result = personal_inventory_cli.format_merge_rows_result
format_owned_rows = personal_inventory_cli.format_owned_rows
format_remove_card_result = personal_inventory_cli.format_remove_card_result
format_set_acquisition_result = personal_inventory_cli.format_set_acquisition_result
format_set_condition_result = personal_inventory_cli.format_set_condition_result
format_set_location_result = personal_inventory_cli.format_set_location_result
format_set_notes_result = personal_inventory_cli.format_set_notes_result
format_set_tags_result = personal_inventory_cli.format_set_tags_result
format_split_row_result = personal_inventory_cli.format_split_row_result
inventory_health = personal_inventory_cli.inventory_health
inventory_report = personal_inventory_cli.inventory_report
list_inventories = personal_inventory_cli.list_inventories
list_owned = personal_inventory_cli.list_owned
merge_rows = personal_inventory_cli.merge_rows
remove_card = personal_inventory_cli.remove_card
render_table = personal_inventory_cli.render_table
search_cards = personal_inventory_cli.search_cards
set_acquisition = personal_inventory_cli.set_acquisition
set_condition = personal_inventory_cli.set_condition
set_location = personal_inventory_cli.set_location
set_notes = personal_inventory_cli.set_notes
set_tags = personal_inventory_cli.set_tags
split_row = personal_inventory_cli.split_row
valuation = personal_inventory_cli.valuation

WORK_DIR = VAR_DIR / "walkthrough"
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True)

DB_PATH = WORK_DIR / "walkthrough.db"
SCRYFALL_JSON = WORK_DIR / "scryfall_sample.json"
IDENTIFIERS_JSON = WORK_DIR / "identifiers_sample.json"
PRICES_JSON = WORK_DIR / "prices_sample.json"
EXPORT_CSV = WORK_DIR / "burn_tag_export.csv"
REPORT_TXT = WORK_DIR / "personal_report.txt"
REPORT_JSON = WORK_DIR / "personal_report.json"
REPORT_ROWS_CSV = WORK_DIR / "personal_report_rows.csv"


def snapshot_preview_rows(snapshots: list[dict]) -> list[dict]:
    preview = []
    for row in snapshots:
        preview.append(
            {
                "snapshot_name": row.get("snapshot_name", ""),
                "label": row.get("label", ""),
                "created_at": str(row.get("created_at", "")),
                "size_bytes": row.get("size_bytes", ""),
            }
        )
    return preview


print("repo root:", REPO_ROOT)
print("src dir:", SRC_DIR)
print("work dir:", WORK_DIR)
print("importer module:", mvp_importer.__file__)
print("inventory cli module:", personal_inventory_cli.__file__)


## Build Tiny Sample Data Files

The real import flow expects local Scryfall and MTGJSON bulk files. For this walkthrough, we write tiny stand-in JSON files that keep the structure the code expects while staying easy to read.

If you want to try a larger or real import later, swap these files for actual bulk downloads and rerun the import sections.


In [ ]:
scryfall_payload = [
    {
        "id": "s1",
        "oracle_id": "o1",
        "name": "Lightning Bolt",
        "set": "lea",
        "set_name": "Limited Edition Alpha",
        "collector_number": "161",
        "lang": "en",
        "rarity": "common",
        "released_at": "1993-08-05",
        "mana_cost": "{R}",
        "type_line": "Instant",
        "oracle_text": "Lightning Bolt deals 3 damage to any target.",
        "colors": ["R"],
        "color_identity": ["R"],
        "finishes": ["nonfoil"],
        "legalities": {"commander": "legal"},
        "purchase_uris": {"tcgplayer": "https://example.test/tcg/lightning-bolt"},
        "tcgplayer_id": 534658,
        "cardmarket_id": 752712,
    },
    {
        "id": "s2",
        "oracle_id": "o2",
        "name": "Sol Ring",
        "set": "clb",
        "set_name": "Commander Legends: Battle for Baldur's Gate",
        "collector_number": "860",
        "lang": "en",
        "rarity": "rare",
        "released_at": "2022-06-10",
        "mana_cost": "{1}",
        "type_line": "Artifact",
        "oracle_text": "{T}: Add {C}{C}.",
        "colors": [],
        "color_identity": [],
        "finishes": ["nonfoil", "foil"],
        "legalities": {"commander": "legal"},
        "purchase_uris": {"tcgplayer": "https://example.test/tcg/sol-ring"},
        "tcgplayer_id": 271111,
        "cardmarket_id": 665555,
    },
    {
        "id": "s3",
        "oracle_id": "o3",
        "name": "Counterspell",
        "set": "7ed",
        "set_name": "Seventh Edition",
        "collector_number": "67",
        "lang": "en",
        "rarity": "uncommon",
        "released_at": "2001-04-11",
        "mana_cost": "{U}{U}",
        "type_line": "Instant",
        "oracle_text": "Counter target spell.",
        "colors": ["U"],
        "color_identity": ["U"],
        "finishes": ["nonfoil"],
        "legalities": {"commander": "legal"},
        "purchase_uris": {"tcgplayer": "https://example.test/tcg/counterspell"},
        "tcgplayer_id": 333333,
        "cardmarket_id": 444444,
    },
]

identifiers_payload = {
    "data": {
        "uuid-1": {
            "identifiers": {
                "scryfallId": "s1",
                "tcgplayerProductId": "534658",
                "cardKingdomId": "12345",
                "mcmId": "752712",
                "cardsphereId": "98765",
            }
        },
        "uuid-2": {
            "identifiers": {
                "scryfallId": "s2",
                "tcgplayerProductId": "271111",
                "cardKingdomFoilId": "54321",
                "mcmId": "665555",
                "cardsphereFoilId": "56789",
            }
        },
        "uuid-3": {
            "identifiers": {
                "scryfallId": "s3",
                "tcgplayerProductId": "333333",
                "cardKingdomId": "99999",
                "mcmId": "444444",
                "cardsphereId": "11111",
            }
        },
    }
}

prices_payload = {
    "data": {
        "uuid-1": {
            "paper": {
                "tcgplayer": {
                    "currency": "USD",
                    "retail": {"normal": {"2026-03-27": 2.92}},
                    "buylist": {"normal": {"2026-03-27": 1.10}},
                },
                "cardmarket": {
                    "currency": "EUR",
                    "retail": {"normal": {"2026-03-27": 1.51}},
                },
            }
        },
        "uuid-2": {
            "paper": {
                "tcgplayer": {
                    "currency": "USD",
                    "retail": {
                        "normal": {"2026-03-27": 1.75},
                        "foil": {"2026-03-27": 4.25},
                    },
                    "buylist": {
                        "normal": {"2026-03-27": 0.75},
                        "foil": {"2026-03-27": 2.00},
                    },
                }
            }
        },
        "uuid-3": {
            "paper": {
                "tcgplayer": {
                    "currency": "USD",
                    "retail": {"normal": {"2026-03-27": 1.25}},
                    "buylist": {"normal": {"2026-03-27": 0.45}},
                }
            }
        },
    }
}

SCRYFALL_JSON.write_text(json.dumps(scryfall_payload), encoding="utf-8")
IDENTIFIERS_JSON.write_text(json.dumps(identifiers_payload), encoding="utf-8")
PRICES_JSON.write_text(json.dumps(prices_payload), encoding="utf-8")

print(SCRYFALL_JSON)
print(IDENTIFIERS_JSON)
print(PRICES_JSON)


## Initialize the Database, Import the Catalog, and Create a Safety Snapshot

This section uses the importer functions directly from Python. That makes the data flow easier to follow inside the notebook while still using the same underlying code that powers the CLI commands.


In [ ]:
initialize_database(DB_PATH)

scryfall_stats = import_scryfall_cards(DB_PATH, SCRYFALL_JSON)
identifier_stats = import_mtgjson_identifiers(DB_PATH, IDENTIFIERS_JSON)
price_stats = import_mtgjson_prices(DB_PATH, PRICES_JSON)
post_import_snapshot = create_database_snapshot(DB_PATH, label="after_bulk_import")

print("scryfall:", scryfall_stats)
print("identifiers:", identifier_stats)
print("prices:", price_stats)
print()
print("Snapshots")
print(render_table(
    snapshot_preview_rows(list_database_snapshots(DB_PATH, limit=5)),
    [
        ("snapshot_name", "snapshot_name"),
        ("label", "label"),
        ("created_at", "created_at"),
        ("size_bytes", "size_bytes"),
    ],
))


## Inspect the Imported Catalog Tables

Before building an inventory, it helps to see what was loaded. The tables below show the normalized card catalog and the price snapshots that later power collection valuation.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    cards = [
        dict(row)
        for row in conn.execute(
            """
            SELECT scryfall_id, mtgjson_uuid, name, set_code, collector_number,
                   tcgplayer_product_id, cardkingdom_id, cardmarket_id
            FROM mtg_cards
            ORDER BY name
            """
        )
    ]
    prices = [
        dict(row)
        for row in conn.execute(
            """
            SELECT scryfall_id, provider, price_kind, finish, currency, snapshot_date, price_value
            FROM price_snapshots
            ORDER BY scryfall_id, provider, price_kind, finish
            """
        )
    ]

print("Cards")
print(render_table(
    cards,
    [
        ("scryfall_id", "scryfall_id"),
        ("mtgjson_uuid", "uuid"),
        ("name", "name"),
        ("set_code", "set"),
        ("collector_number", "number"),
        ("tcgplayer_product_id", "tcgplayer"),
        ("cardkingdom_id", "cardkingdom"),
        ("cardmarket_id", "cardmarket"),
    ],
))
print()
print("Prices")
print(render_table(
    prices,
    [
        ("scryfall_id", "scryfall_id"),
        ("provider", "provider"),
        ("price_kind", "kind"),
        ("finish", "finish"),
        ("currency", "currency"),
        ("snapshot_date", "date"),
        ("price_value", "price_value"),
    ],
))


## Create a Personal Inventory and Search the Local Catalog

The catalog tables store canonical printings and price history. Inventory rows store the copies you own and link each row back to an exact printing.

This separation is what lets the repo support both catalog search and collection valuation from the same local database.


In [ ]:
create_inventory(
    DB_PATH,
    slug="personal",
    display_name="Personal Collection",
    description="Binders, decks, and loose cards",
)

inventory_rows = list_inventories(DB_PATH)
print("Inventories")
print(render_table(
    inventory_rows,
    [
        ("slug", "slug"),
        ("display_name", "name"),
        ("description", "description"),
        ("item_rows", "item_rows"),
        ("total_cards", "total_cards"),
    ],
))
print()

search_columns = [
    ("name", "name"),
    ("set_code", "set"),
    ("set_name", "set_name"),
    ("collector_number", "number"),
    ("lang", "lang"),
    ("rarity", "rarity"),
    ("finishes", "finishes"),
    ("scryfall_id", "scryfall_id"),
]

bolt_matches = search_cards(
    DB_PATH,
    query="Lightning Bolt",
    set_code="lea",
    rarity="common",
    finish="normal",
    lang="en",
    exact=False,
    limit=10,
)
print("Lightning Bolt matches")
print(render_table(bolt_matches, search_columns))
print()

counterspell_matches = search_cards(
    DB_PATH,
    query="Counterspell",
    set_code=None,
    rarity="uncommon",
    finish=None,
    lang="en",
    exact=True,
    limit=10,
)
print("Counterspell matches")
print(render_table(counterspell_matches, search_columns))


## Add Owned Cards

Here we add a few owned cards with quantity, condition, finish, acquisition info, location, notes, and tags.

The output uses the same human-readable format a terminal user would see after running the CLI.


In [ ]:
bolt_result = add_card(
    DB_PATH,
    inventory_slug="personal",
    scryfall_id="s1",
    name=None,
    set_code=None,
    collector_number=None,
    lang=None,
    quantity=4,
    condition_code="NM",
    finish="normal",
    language_code="en",
    location="Red Binder",
    acquisition_price=2.00,
    acquisition_currency="USD",
    notes="Playset",
    tags="burn,trade",
)

ring_result = add_card(
    DB_PATH,
    inventory_slug="personal",
    scryfall_id="s2",
    name=None,
    set_code=None,
    collector_number=None,
    lang=None,
    quantity=1,
    condition_code="LP",
    finish="foil",
    language_code="en",
    location="Commander Deck Box",
    acquisition_price=3.50,
    acquisition_currency="USD",
    notes="One shiny copy",
    tags=None,
)

counterspell_result = add_card(
    DB_PATH,
    inventory_slug="personal",
    scryfall_id="s3",
    name=None,
    set_code=None,
    collector_number=None,
    lang=None,
    quantity=2,
    condition_code="NM",
    finish="normal",
    language_code="en",
    location="",
    acquisition_price=1.00,
    acquisition_currency="USD",
    notes=None,
    tags=None,
)

print(format_add_card_result(bolt_result))
print()
print(format_add_card_result(ring_result))
print()
print(format_add_card_result(counterspell_result))


## Day-to-Day Editing Commands

Once cards are in an inventory, most work is small maintenance: updating tags and notes, correcting storage locations, changing condition, recording acquisition cost, and moving quantity between rows.

This section walks through those common edits, including splitting and merging inventory rows.


In [ ]:
owned_rows = list_owned(DB_PATH, inventory_slug="personal", provider="tcgplayer", limit=None)
ring_row = next(row for row in owned_rows if row["name"] == "Sol Ring")
counterspell_row = next(row for row in owned_rows if row["name"] == "Counterspell")

ring_tags_result = set_tags(
    DB_PATH,
    inventory_slug="personal",
    item_id=ring_row["item_id"],
    tags="commander,foil",
)
ring_notes_result = set_notes(
    DB_PATH,
    inventory_slug="personal",
    item_id=ring_row["item_id"],
    notes="Needs fresh sleeve",
)
ring_location_result = set_location(
    DB_PATH,
    inventory_slug="personal",
    item_id=ring_row["item_id"],
    location="Commander Deck Box / Main",
)
ring_condition_result = set_condition(
    DB_PATH,
    inventory_slug="personal",
    item_id=ring_row["item_id"],
    condition_code="NM",
)
ring_acquisition_result = set_acquisition(
    DB_PATH,
    inventory_slug="personal",
    item_id=ring_row["item_id"],
    acquisition_price=4.00,
    acquisition_currency="USD",
)

counterspell_split_result = split_row(
    DB_PATH,
    inventory_slug="personal",
    item_id=counterspell_row["item_id"],
    quantity=1,
    condition_code=None,
    finish=None,
    language_code=None,
    location="Blue Deck",
)

split_row_tag_result = set_tags(
    DB_PATH,
    inventory_slug="personal",
    item_id=counterspell_split_result["item_id"],
    tags="control",
)

counterspell_merge_result = merge_rows(
    DB_PATH,
    inventory_slug="personal",
    source_item_id=counterspell_split_result["item_id"],
    target_item_id=counterspell_split_result["source_item_id"],
)

print(format_set_tags_result(ring_tags_result))
print()
print(format_set_notes_result(ring_notes_result))
print()
print(format_set_location_result(ring_location_result))
print()
print(format_set_condition_result(ring_condition_result))
print()
print(format_set_acquisition_result(ring_acquisition_result))
print()
print(format_split_row_result(counterspell_split_result))
print()
print(format_set_tags_result(split_row_tag_result))
print()
print(format_merge_rows_result(counterspell_merge_result))
print()
print("Inventory after editing")
print(format_owned_rows(list_owned(DB_PATH, inventory_slug="personal", provider="tcgplayer", limit=None)))


## Review Inventory Health, Export CSV, and Build a Report

After the inventory has some real data, the reporting tools help answer three practical questions:

- What do I own and what is it worth?
- Which rows are missing useful metadata such as tags or locations?
- How can I export or summarize the collection for other tools?

The cells below show the inventory health report, a filtered CSV export, and a broader inventory summary report.


In [ ]:
owned_rows = list_owned(DB_PATH, inventory_slug="personal", provider="tcgplayer", limit=None)
print("Owned rows")
print(format_owned_rows(owned_rows))
print()

valuation_rows = valuation(DB_PATH, inventory_slug="personal", provider="tcgplayer")
print("Valuation totals")
print(render_table(
    valuation_rows,
    [
        ("provider", "provider"),
        ("currency", "currency"),
        ("item_rows", "item_rows"),
        ("total_cards", "total_cards"),
        ("total_value", "total_value"),
    ],
))
print()

health_result = inventory_health(
    DB_PATH,
    inventory_slug="personal",
    provider="tcgplayer",
    stale_days=30,
    preview_limit=10,
)
print(format_inventory_health_result(health_result))
print()

export_result = export_inventory_csv(
    DB_PATH,
    inventory_slug="personal",
    provider="tcgplayer",
    output_path=EXPORT_CSV,
    query=None,
    set_code=None,
    rarity=None,
    finish=None,
    condition_code=None,
    language_code=None,
    location=None,
    tags=["burn"],
    limit=None,
)
print(format_export_csv_result(export_result))
print()
print("Export preview")
print("\n".join(EXPORT_CSV.read_text(encoding="utf-8").splitlines()[:6]))
print()

report_result = inventory_report(
    DB_PATH,
    inventory_slug="personal",
    provider="tcgplayer",
    query=None,
    set_code=None,
    rarity=None,
    finish=None,
    condition_code=None,
    language_code=None,
    location=None,
    tags=None,
    limit=10,
    stale_days=30,
)
print(format_inventory_report_result(report_result))


## Run the CLI from the Notebook

So far the notebook has used the Python API directly. This section switches to the command-line interface to show how a non-Python user would generate report files.

Inside the notebook we call `python -m ...` so the command runs against this checkout of the repo. In a normal terminal session, the installed commands from the README are the usual entry point.


In [ ]:
report_cmd = [
    sys.executable,
    "-m",
    "mtg_source_stack.personal_inventory_cli",
    "inventory-report",
    "--db",
    str(DB_PATH),
    "--inventory",
    "personal",
    "--provider",
    "tcgplayer",
    "--report-out",
    str(REPORT_TXT),
    "--report-out-json",
    str(REPORT_JSON),
    "--report-out-csv",
    str(REPORT_ROWS_CSV),
]

completed = subprocess.run(
    report_cmd,
    cwd=REPO_ROOT,
    env=CLI_ENV,
    check=True,
    capture_output=True,
    text=True,
)

print(completed.stdout)
print()
for path in (REPORT_TXT, REPORT_JSON, REPORT_ROWS_CSV):
    print(f"{path.name}: {path.exists()} -> {path}")


## Snapshot Rollback Demonstration

Safety snapshots are an important part of the local-first workflow. They give you a quick way to recover from mistakes before or after a destructive change.

This section creates a snapshot, removes a row, and then restores the database to prove the inventory can be recovered.


In [ ]:
snapshot_before_remove = create_database_snapshot(DB_PATH, label="before_remove_demo")
print("Snapshots before remove")
print(render_table(
    snapshot_preview_rows(list_database_snapshots(DB_PATH, limit=5)),
    [
        ("snapshot_name", "snapshot_name"),
        ("label", "label"),
        ("created_at", "created_at"),
        ("size_bytes", "size_bytes"),
    ],
))
print()

owned_rows = list_owned(DB_PATH, inventory_slug="personal", provider="tcgplayer", limit=None)
ring_row = next(row for row in owned_rows if row["name"] == "Sol Ring")
removed_result = remove_card(
    DB_PATH,
    inventory_slug="personal",
    item_id=ring_row["item_id"],
)
print(format_remove_card_result(removed_result))
print()
print("Rows after removal")
print(format_owned_rows(list_owned(DB_PATH, inventory_slug="personal", provider="tcgplayer", limit=None)))
print()

restore_result = restore_database_snapshot(
    DB_PATH,
    snapshot=snapshot_before_remove["snapshot_name"],
)
print("Restored snapshot:", restore_result["snapshot_name"])
if restore_result["pre_restore_snapshot"] is not None:
    print("Pre-restore safety snapshot:", restore_result["pre_restore_snapshot"]["snapshot_name"])
print()
print("Rows after restore")
print(format_owned_rows(list_owned(DB_PATH, inventory_slug="personal", provider="tcgplayer", limit=None)))


## What This Notebook Demonstrated

You now have a complete small-scale tour of the repo's main workflow:

- importing catalog and price data into SQLite
- building and searching a personal inventory
- editing owned rows with tags, notes, location, condition, acquisition, split, and merge actions
- checking collection health and generating exports and reports
- rolling back changes with snapshots

Good next steps:

- read the repo overview in `README.md`
- inspect `examples/sample_inventory_import.csv` for CSV import shape
- use `docs/ingestion_flow.md` and `docs/source_map.md` for deeper architecture context
- rerun this notebook with your own sample data once you're comfortable with the flow
